# Text as data

A statistical model needs numbers, but a customs decision is a string of characters: a description of goods in German, French, Polish or one of 19 other languages. This page shows the first and most important step of every text model: turning documents into a table of numbers. It covers the vocabulary of a corpus, the usual preprocessing steps and what they mean for a multilingual corpus, the bag-of-words representation computed by hand and with scikit-learn, TF-IDF weights and n-grams, the sparse matrices that make this representation fit into memory, and simple statistics of a text that can serve as features without any vocabulary. The choices made here decide what any later model can and cannot see, from the linear classifiers of Block 2 to the language models of Session 14.

**The case study in one paragraph.** Traders who are unsure how their product is classified in the EU customs tariff can ask a customs authority for a Binding Tariff Information (BTI) decision; customs describes the product and states its code, and the European Commission publishes all decisions in the EBTI database. The code follows the Harmonized System (HS): 21 sections, 97 chapters (2 digits), about 1,200 headings (4 digits) and finer subheadings. Our task is to predict the **four-digit heading** from the **description of goods**. This session starts the part of the course that works with these decisions (Sessions 13–16, with the leaderboard rounds L1, L2 and L3); Block 2 sets out the task, the time-based split and the leaderboard ([lecture notebook 03](03-lecture-text-classification-and-error-analysis.ipynb)), and [case-study/README.md](../../../case-study/README.md) describes the data.

The code cells in this notebook build on each other; run them in order (the first cell moves to the repository root; see the session [README](../README.md#setup) for the environment). The n-gram cell takes about half a minute.

```mermaid
flowchart LR
    A["Description<br/>of goods"] --> B["Normalise<br/>(lower-case)"]
    B --> C["Tokenise<br/>(words or characters)"]
    C --> D["Filter / reduce<br/>(stop words, stemming)"]
    D --> E["Vocabulary<br/>(fit on training data)"]
    E --> F["Count and weight<br/>(counts, TF-IDF)"]
    F --> G["Document-term matrix<br/>(sparse)"]
    G --> H["Model<br/>(Block 2)"]
```

In [1]:
# Run the code from the repository root, where the case-study paths start.
import os
from pathlib import Path

os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir()))

## Documents, tokens and vocabulary

### Concept

A **corpus** is a collection of texts. Each text in it is a **document**; in the case study one document is the description of goods of one decision. A **token** is one unit of text after splitting, usually a word, sometimes a number or a punctuation mark. A **type** is a distinct token: in "toy car made of plastic, plastic car" there are seven tokens but only five types (`toy`, `car`, `made`, `of`, `plastic`). The **vocabulary** is the set of types the model knows, and each type gets one fixed column number.

Worked example, by hand:

| Document | Tokens |
|---|---|
| d1 = "Toy car made of plastic." | toy, car, made, of, plastic |
| d2 = "Car seat cover made of textile." | car, seat, cover, made, of, textile |

Together the two documents have 11 tokens and 8 types. Sorted alphabetically, the vocabulary is `car, cover, made, of, plastic, seat, textile, toy`; `car` is column 0 and `toy` column 7.

In scikit-learn the vocabulary is learned by `fit` and applied by `transform`. A word that did not occur during `fit` has no column, so it is silently ignored later. This is the **out-of-vocabulary** problem: a new product name, a misspelling, or a word in a language that was rare in the training data is invisible to the model.

### Why it matters

Every text model starts by splitting text into tokens and mapping them to numbers. If the vocabulary is learned from the wrong data (for example from training and test decisions together), information leaks from the test set. If it is learned from too little data, many words in new documents are out of vocabulary; with 22 languages this happens quickly. The size of the vocabulary also sets the number of model parameters: a linear model has one weight per vocabulary entry and class, and the case study has about 1,100 classes.

### How it works in Python

In [2]:
from collections import Counter

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

docs = ["Toy car made of plastic.", "Car seat cover made of textile."]
tokens = [d.lower().replace(".", "").split() for d in docs]
print(tokens[1])                          # ['car', 'seat', 'cover', 'made', 'of', 'textile']
print(Counter(tokens[0] + tokens[1]).most_common(3))   # [('car', 2), ('made', 2), ('of', 2)]
vocab = sorted(set(tokens[0]) | set(tokens[1]))
print(len(vocab), vocab)   # 8 ['car', 'cover', 'made', 'of', 'plastic', 'seat', 'textile', 'toy']

cv = CountVectorizer()                    # the same steps in one object
cv.fit(docs)                              # learn the vocabulary
print(cv.vocabulary_["toy"])              # 7: the column of 'toy'
print(cv.transform(["Toy truck made of wood."]).toarray())
# [[0 0 1 1 0 0 0 1]]: 'truck' and 'wood' are out of vocabulary and simply disappear

# the case-study corpus: one document = one description of goods
decisions = pd.read_parquet("case-study/data/train_sample.parquet")
texts = decisions["description"]
n_tokens = texts.str.split().str.len()
print(len(texts), int(n_tokens.sum()))    # 50000 documents, 4504393 tokens
print(n_tokens.median())                  # 82.0 tokens in a typical description
print(texts.str.len().groupby(decisions["language"]).median().loc[["fr", "en", "de"]].to_dict())
# {'fr': 272.0, 'en': 309.0, 'de': 740.0}: German descriptions are more than twice as long

['car', 'seat', 'cover', 'made', 'of', 'textile']
[('car', 2), ('made', 2), ('of', 2)]
8 ['car', 'cover', 'made', 'of', 'plastic', 'seat', 'textile', 'toy']
7
[[0 0 1 1 0 0 0 1]]


50000 4504393
82.0
{'fr': 272.0, 'en': 309.0, 'de': 740.0}


### In practice

- Search engines such as Elasticsearch and Apache Lucene pass every indexed document through an *analyzer* that tokenises and normalises it, with one analyzer per language; the query goes through the same analyzer so that query tokens and document tokens match.
- Customs administrations work on the same task as this course: the World Customs Organization's data-analytics initiative (BACUDA) and several national administrations have built models that suggest HS codes from the free-text goods descriptions of declarations, so that officers check suggestions instead of searching the tariff.
- Mosteller and Wallace (1964) attributed the disputed *Federalist Papers* to James Madison by comparing how often the candidate authors used common words such as "upon" and "whilst", one of the earliest statistical studies that treated text as counts.

> **Warning.** **Fit the vocabulary on training data only.** If `fit` sees the validation or test documents, the vocabulary (and later the TF-IDF weights) contain information from data the model should not have seen. Put the vectoriser inside a scikit-learn `Pipeline`, so that cross-validation refits it on each training fold (Session 7).

## Preprocessing: lower-casing, stop words, stemming and lemmatisation

### Concept

Preprocessing maps different surface forms to one form, so that the vocabulary becomes smaller and counts become more reliable.

- **Normalisation** maps different spellings to one form. The most common step is **lower-casing** ("Kunststoff", "KUNSTSTOFF" → "kunststoff"). Others are removing accents, HTML tags or line breaks. Some decisions are written entirely in capitals, so lower-casing matters here.
- **Tokenisation** splits text into tokens. scikit-learn's default pattern `(?u)\b\w\w+\b` keeps runs of two or more letters or digits. Numbers such as `220` (volts) or `100` (per cent) become tokens; single characters disappear.
- **Stop words** are very frequent function words ("the", "and", "aus", "mit", "de", "la") that carry little content. scikit-learn ships only an **English** list of 318 words. In the case study, where 57 % of the descriptions are German and 16 % French, this list removes almost nothing.
- **Stemming** cuts word endings with fixed rules. The Snowball stemmers (an extension of Porter, 1980) exist for German, French, Dutch, Swedish and other languages and map "Spielzeuge", "Spielzeugen" to `spielzeug`. A stem need not be a real word ("chaussures" → `chaussur`).
- **Lemmatisation** looks a word up together with its part of speech and returns its dictionary form, the **lemma**. It needs a trained pipeline per language (spaCy has German, French, Dutch, Polish and others), is slower than stemming and more accurate.

A special problem of German, Dutch and Swedish is **compounding**: "Kinderspielzeug", "Kunststoffspielzeug" and "Holzspielzeug" (children's, plastic and wooden toy) are three separate tokens that share no column with "Spielzeug". Stemming does not help, because the stem is the whole compound.

Worked example for the German description "Kinderspielzeug aus Kunststoff, in Form eines Autos, mit Rädern aus Gummi":

| Step | Result |
|---|---|
| lower-case and tokenise | kinderspielzeug, aus, kunststoff, in, form, eines, autos, mit, rädern, aus, gummi |
| remove English stop words | the same except `in`: the English list does not know German |
| remove a German stop-word list | kinderspielzeug, kunststoff, form, autos, rädern, gummi |
| German stemming | kinderspielzeug, kunststoff, form, autos, rad, gummi (the compound stays whole; the loan word "autos" is not reduced) |

| Step | Vocabulary | Information lost | Cost |
|---|---|---|---|
| lower-casing | smaller | little (German nouns are capitalised, but the word stays) | none |
| stop-word removal | slightly smaller | "ohne" (without), "nicht" (not), "pour" (for) can matter for a tariff | needs one list per language |
| stemming | smaller | distinct words may merge | one stemmer per language |
| lemmatisation | smaller | little | one language model per language; slow |
| character n-grams (below) | much larger | none; replaces most of the above | more features |

```mermaid
flowchart TD
    Q{"One language<br/>or many?"} -->|one| S{"Labelled data<br/>scarce?"}
    Q -->|many| C["Keep words as they are<br/>and add character n-grams<br/>(no per-language rules)"]
    S -->|no| K["Keep the defaults:<br/>lower-case only"]
    S -->|yes| L["Try stemming or<br/>lemmatisation"]
    C --> V["Compare on validation data"]
    K --> V
    L --> V
```

### Why it matters

Preprocessing reduces the number of columns and merges forms that mean the same thing, which helps when labelled data are scarce. Every step also removes information, and in a multilingual corpus every rule-based step must exist once per language. Character n-grams (section on n-grams below) are the pragmatic alternative: they need no language-specific rules and share pieces such as `spiel` and `zeug` between compounds. Whether a step helps is an empirical question: test it on validation data instead of applying it by habit.

### How it works in Python

In [3]:
import re

from nltk.stem.snowball import SnowballStemmer      # pure Python; no data download needed
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

text = "Kinderspielzeug aus Kunststoff, in Form eines Autos, mit Rädern aus Gummi"
tokens = re.findall(r"(?u)\b\w\w+\b", text.lower())     # scikit-learn's default token pattern
print(tokens)
# ['kinderspielzeug', 'aus', 'kunststoff', 'in', 'form', 'eines', 'autos', 'mit', 'rädern', 'aus', 'gummi']
print([t for t in tokens if t not in ENGLISH_STOP_WORDS])   # only 'in' is removed
german_stop = {"aus", "in", "eines", "mit", "der", "die", "das", "und", "einem", "einer"}
print([t for t in tokens if t not in german_stop])
# ['kinderspielzeug', 'kunststoff', 'form', 'autos', 'rädern', 'gummi']

de = SnowballStemmer("german")
print([de.stem(w) for w in ["Spielzeug", "Spielzeuge", "Spielzeugen", "Kunststoff", "Kunststoffen"]])
# ['spielzeug', 'spielzeug', 'spielzeug', 'kunststoff', 'kunststoff']
fr = SnowballStemmer("french")
print([fr.stem(w) for w in ["jouet", "jouets", "chaussure", "chaussures"]])
# ['jouet', 'jouet', 'chaussur', 'chaussur']

# effect on the vocabulary of the case-study sample
for name, kwargs in [("default", {}), ("no lower-casing", {"lowercase": False}),
                     ("English stop words removed", {"stop_words": "english"})]:
    print(name, len(CountVectorizer(**kwargs).fit(texts).vocabulary_))
# default 213583 / no lower-casing 237686 / English stop words removed 213336

# compounds: character 4-grams shared by 'Kunststoffspielzeug' and 'Spielzeug'
grams = CountVectorizer(analyzer="char_wb", ngram_range=(4, 4)).build_analyzer()
print(sorted(set(grams("Kunststoffspielzeug")) & set(grams("Spielzeug"))))
# ['elze', 'eug ', 'ielz', 'lzeu', 'piel', 'spie', 'zeug']: the compound and the simple word now overlap

['kinderspielzeug', 'aus', 'kunststoff', 'in', 'form', 'eines', 'autos', 'mit', 'rädern', 'aus', 'gummi']
['kinderspielzeug', 'aus', 'kunststoff', 'form', 'eines', 'autos', 'mit', 'rädern', 'aus', 'gummi']
['kinderspielzeug', 'kunststoff', 'form', 'autos', 'rädern', 'gummi']
['spielzeug', 'spielzeug', 'spielzeug', 'kunststoff', 'kunststoff']
['jouet', 'jouet', 'chaussur', 'chaussur']


default 213583


no lower-casing 237686


English stop words removed 213336
['elze', 'eug ', 'ielz', 'lzeu', 'piel', 'spie', 'zeug']


The English stop-word list removes 247 of 213,583 word types: it was made for English text.

### In practice

- Elasticsearch offers stemmer, stop-word and *decompounder* token filters for German, Dutch and the Scandinavian languages, so that a query for "Spielzeug" also finds "Kinderspielzeug".
- Machine-translation and search systems for the EU institutions handle all 24 official languages; the EU's IATE terminology database links the same concept across languages, which rule-based preprocessing cannot do on its own.
- The SpaCy and Stanza libraries provide tokenisers and lemmatisers for dozens of languages, which is why multilingual projects often start with them rather than with hand-written rules.

> **Caution.** `stop_words="english"` is a fixed English list. Applied to German or French text it does almost nothing; applied to English text it also removes "not", "no", "without" and "nor". "Shoes without laces" and "shoes with laces" can belong to different subheadings. If you remove stop words at all, use your own list per language and check it.

> **Tip.** The descriptions also contain line breaks (`\r\n`), measurements ("30 cm", "220 V") and the placeholder `<CODE>` where a number repeated the decision's own code. The default tokeniser turns `<CODE>` into the token `code`; keep this in mind when you read the most important features in Block 2.

> 📖 **Read more:** [extras/01-data100-text-wrangling-regex.ipynb](extras/01-data100-text-wrangling-regex.ipynb) (Data 100): canonicalising and extracting text with Python and Polars string methods and regular expressions (character classes, greediness, capture groups), the hand-written cleaning that comes before tokenisation.

## Bag-of-words computed by hand

### Concept

The **bag-of-words** representation describes a document only by how often each vocabulary word occurs in it. The order of the words is thrown away, as if the words were shaken in a bag. Stacking the count vectors of all documents gives the **document-term matrix** (DTM): one row per document, one column per vocabulary word, and in cell (i, j) the count of word j in document i.

Worked example with three short descriptions:

- d1 = "toy car, plastic car"
- d2 = "plastic box with lid"
- d3 = "toy box made of wood"

Step 1: tokenise and collect the vocabulary in alphabetical order: `box, car, lid, made, of, plastic, toy, with, wood` (9 types).

Step 2: count each word in each document.

| | box | car | lid | made | of | plastic | toy | with | wood |
|---|---|---|---|---|---|---|---|---|---|
| d1 | 0 | 2 | 0 | 0 | 0 | 1 | 1 | 0 | 0 |
| d2 | 1 | 0 | 1 | 0 | 0 | 1 | 0 | 1 | 0 |
| d3 | 1 | 0 | 0 | 1 | 1 | 0 | 1 | 0 | 1 |

Step 3: read the table. d1 and d2 share `plastic`; d2 and d3 share `box`; d1 and d3 share `toy`. The row sums (4, 4, 5) are the document lengths in tokens. The column sums are the corpus frequencies of the words. Most cells are zero even in this tiny example (15 of 27).

The figure shows the same idea for five short descriptions in three languages.

![Heatmap of a document-term matrix with five short descriptions in English, German and French as rows and the vocabulary words as columns; most cells are zero and the languages share almost no columns](../figures/dtm_sketch.png)

A bag-of-words vector is a point in a space with one axis per word. Two documents are similar if they use the same words in similar proportions. The figure shows the consequence for a multilingual corpus: an English and a German description of the same toy share no word and therefore look unrelated. A classifier can still learn both, but only from labelled examples in each language.

### Why it matters

Bag-of-words is simple, fast and transparent: every column is a word you can read, and every model weight can be traced back to a word. Descriptions of goods are well suited, because the decisive information is mostly in nouns and materials ("Schuhe", "Leder", "Kunststoff"). Its blind spots (word order, synonyms, other languages) are the subject of the last section of Block 2 and the motivation for Session 14.

### How it works in Python

In [4]:
mini = ["toy car, plastic car", "plastic box with lid", "toy box made of wood"]
cv = CountVectorizer()
X = cv.fit_transform(mini)
print(cv.get_feature_names_out())
# ['box' 'car' 'lid' 'made' 'of' 'plastic' 'toy' 'with' 'wood']
print(X.toarray())
# [[0 2 0 0 0 1 1 0 0]
#  [1 0 1 0 0 1 0 1 0]
#  [1 0 0 1 1 0 1 0 1]]
print(X.sum(axis=1).A1, X.sum(axis=0).A1)   # row sums [4 4 5], column sums [2 2 1 1 1 2 2 1 1]

# bag-of-words ignores order: these two descriptions get the same vector
same = cv.transform(["plastic car, toy car", "toy car, plastic car"]).toarray()
print((same[0] == same[1]).all())            # True

# binary=True records presence instead of counts (useful for short texts)
print(CountVectorizer(binary=True).fit_transform(mini).toarray()[0])  # [0 1 0 0 0 1 1 0 0]

['box' 'car' 'lid' 'made' 'of' 'plastic' 'toy' 'with' 'wood']
[[0 2 0 0 0 1 1 0 0]
 [1 0 1 0 0 1 0 1 0]
 [1 0 0 1 1 0 1 0 1]]
[4 4 5] [2 2 1 1 1 2 2 1 1]
True
[0 1 0 0 0 1 1 0 0]


### In practice

- Spam filters have classified e-mail from word counts since the late 1990s; Paul Graham's essay *A Plan for Spam* (2002) popularised naive Bayes on token counts for this task.
- Slapin and Proksch (2008) developed *Wordfish*, which places German parties on a political scale using only the word counts of their manifestos.
- Automatic coding of occupations and industries in surveys and censuses (for example by the US Census Bureau and national statistics offices) started from bag-of-words matching of free-text answers against coding indexes, the same kind of problem as matching a description of goods to a tariff heading.

> **Note.** "Bag-of-words" does not mean "single words only". The columns can also be pairs of words (bigrams, below) or character sequences. What makes it a bag is that the position of each unit in the document is ignored.

## TF-IDF: weighting words by how informative they are

### Concept

**Term frequency** tf(t, d) is the count of term t in document d. **Document frequency** df(t) is the number of documents that contain t at least once. With N documents, the **inverse document frequency** is

  idf(t) = log(N / df(t)).

It is large for rare terms and zero for a term that occurs in every document. **TF-IDF** is the product tf(t, d) × idf(t): a term gets a high weight in a document if it is frequent there and rare elsewhere.

Worked example with N = 3 documents: d1 = "plastic toy plastic box", d2 = "plastic bag", d3 = "wooden toy".

| Term | tf in d1 | df | idf = ln(3 / df) | tf × idf |
|---|---|---|---|---|
| plastic | 2 | 2 | ln(1.5) = 0.405 | 0.811 |
| toy | 1 | 2 | 0.405 | 0.405 |
| box | 1 | 1 | ln(3) = 1.099 | 1.099 |

"box" occurs once in d1, but it gets the highest weight because no other document uses it.

scikit-learn's `TfidfVectorizer` makes three changes to this textbook formula:

1. **Smoothed idf**: idf(t) = ln((1 + N) / (1 + df(t))) + 1. It acts as if one extra document contained every term, so no division by zero occurs, and the "+ 1" keeps terms that appear everywhere from being removed entirely. For the example: plastic and toy get ln(4/3) + 1 = 1.288, box gets ln(4/2) + 1 = 1.693.
2. **L2 normalisation**: each row is divided by its Euclidean length, so that every document vector has length 1. Long German and short French descriptions become comparable, and the dot product of two rows is their cosine similarity. For d1: the raw weights (2 × 1.288, 1.288, 1.693) = (2.576, 1.288, 1.693) have length 3.34, giving (0.771, 0.386, 0.507).
3. **Sublinear tf** (optional, `sublinear_tf=True`): tf is replaced by 1 + ln(tf). The tenth "Kunststoff" in a description then adds much less than the second.

### Why it matters

Raw counts are dominated by frequent words that carry little information, and in a multilingual corpus there is a separate set of such words for every language. TF-IDF down-weights all of them automatically, without a stop-word list per language, and the row normalisation removes the effect of description length (German descriptions are more than twice as long as French ones). TF-IDF is the standard input for linear text classifiers and was the core of classic keyword search.

### How it works in Python

In [5]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

docs = ["plastic toy plastic box", "plastic bag", "wooden toy"]
# by hand for document 1, with scikit-learn's smoothed idf (N = 3 documents)
tf = {"plastic": 2, "toy": 1, "box": 1}             # counts in document 1
df = {"plastic": 2, "toy": 2, "box": 1}             # documents that contain the word
idf = {w: np.log((1 + 3) / (1 + df[w])) + 1 for w in tf}
print({w: round(float(v), 3) for w, v in idf.items()})   # plastic 1.288, toy 1.288, box 1.693
w = np.array([tf[t] * idf[t] for t in tf])
print((w / np.linalg.norm(w)).round(3))             # [0.771 0.386 0.507] after L2 normalisation

tfidf = TfidfVectorizer()
T = tfidf.fit_transform(docs)
print(tfidf.get_feature_names_out())                # ['bag' 'box' 'plastic' 'toy' 'wooden']
print(T.toarray().round(3)[0])                      # [0.    0.507 0.771 0.386 0.   ]
print(np.linalg.norm(T.toarray(), axis=1))          # [1. 1. 1.]: every row has length 1

# on the case-study sample: which words get the lowest and highest idf?
vec = TfidfVectorizer(min_df=3).fit(texts)
idf_s = pd.Series(vec.idf_, index=vec.get_feature_names_out()).sort_values()
print(idf_s.index[:8].tolist())     # ['in', 'mit', 'aus', 'und', 'der', 'als', 'einem', 'von']
print(idf_s.round(2).iloc[[0, -1]].to_dict())       # {'in': 1.54, 'blaubeere': 10.43}

{'plastic': 1.288, 'toy': 1.288, 'box': 1.693}
[0.771 0.386 0.507]
['bag' 'box' 'plastic' 'toy' 'wooden']
[0.    0.507 0.771 0.386 0.   ]
[1. 1. 1.]


['in', 'mit', 'aus', 'und', 'der', 'als', 'einem', 'von']
{'in': 1.54, 'blaubeere': 10.43}


The lowest idf values belong to German function words, because German is the most frequent language; the highest go to words that occur in exactly three descriptions, such as "Blaubeere" (blueberry).

### In practice

- TF-IDF ranking was the core of classic search engines. Apache Lucene, on which Elasticsearch and Solr are built, used a TF-IDF similarity by default until version 6 (2016), when it switched to BM25, a refinement of the same idea with saturating term frequency and length normalisation (Session 15 uses BM25).
- Keyword extraction: the terms with the highest TF-IDF weight in a document summarise what distinguishes it from the rest of the corpus. Digital libraries and newsrooms use this to tag documents.
- Spärck Jones (1972) introduced the idea of weighting terms by their inverse document frequency in a paper on information retrieval; it is still the default weighting in many retrieval systems.

> **Note.** TF-IDF is still a bag-of-words representation: it changes the *values* in the document-term matrix, not its columns. The matrix keeps the same shape and the same sparsity.

## n-grams: keeping short phrases

### Concept

An **n-gram** is a sequence of n consecutive tokens. Unigrams are single words, **bigrams** are pairs ("leather upper"), **trigrams** are triples. `ngram_range=(1, 2)` adds all bigrams to the vocabulary next to the unigrams.

Worked example from the footwear chapter: heading 6403 covers shoes with uppers of leather, heading 6404 shoes with uppers of textile. "leather upper, textile sole" and "textile upper, leather sole" contain the same four words. With unigrams their vectors are identical. With bigrams the first contains `leather upper` and `textile sole`, the second `textile upper` and `leather sole`, so a model can learn that the material of the *upper* decides the heading.

The cost is size. The number of distinct bigrams grows much faster than the number of words. In the case-study sample with `min_df=3` there are about 67,000 unigrams, 236,000 features with bigrams and 420,000 with trigrams.

**Character n-grams** use sequences of letters instead of words. With `analyzer="char_wb"` scikit-learn builds them inside word boundaries: "zeug" gives ` zeu`, `zeug`, `eug `. They are robust to misspellings and inflections and, most importantly here, they let a German compound such as "Kunststoffspielzeug" share features with "Spielzeug" and "Kunststoff". The price is many more features per document and longer training.

### Why it matters

For tariff classification, short phrases ("aus Leder", "en matière plastique", "for children") and word pieces (material and product names inside compounds) carry the decisive information. On the case-study sample, word bigrams do *not* help (validation accuracy 0.769 against 0.772 for unigrams, measured in [Block 2](03-lecture-text-classification-and-error-analysis.ipynb)), while character n-grams do (0.788). This is an empirical result for this data, not a general rule: Wang and Manning (2012) found bigrams useful for English sentiment, where negations matter. Always measure.

### How it works in Python

In [6]:
pair = ["leather upper, textile sole", "textile upper, leather sole"]
uni = CountVectorizer().fit(pair)
print(uni.transform(pair).toarray())       # [[1 1 1 1] [1 1 1 1]]: identical vectors
bi = CountVectorizer(ngram_range=(1, 2)).fit(pair)
print(bi.get_feature_names_out())
# ['leather' 'leather sole' 'leather upper' 'sole' 'textile' 'textile sole'
#  'textile upper' 'upper' 'upper leather' 'upper textile']
print(bi.transform(pair).toarray())
# [[1 0 1 1 1 1 0 1 0 1]
#  [1 1 0 1 1 0 1 1 1 0]]: now different

for rng in [(1, 1), (1, 2), (1, 3)]:
    v = TfidfVectorizer(ngram_range=rng, min_df=3).fit(texts)
    print(rng, len(v.vocabulary_))         # (1, 1) 66997 / (1, 2) 236112 / (1, 3) 419615

# character n-grams link a compound to its parts
ch = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5)).fit(["Kunststoffspielzeug"])
print(ch.transform(["Spielzeugauto"]).nnz)  # 18 of the 36 character n-grams of 'Spielzeugauto' are known

[[1 1 1 1]
 [1 1 1 1]]
['leather' 'leather sole' 'leather upper' 'sole' 'textile' 'textile sole'
 'textile upper' 'upper' 'upper leather' 'upper textile']
[[1 0 1 1 1 1 0 1 0 1]
 [1 1 0 1 1 0 1 1 1 0]]


(1, 1) 66997


(1, 2) 236112


(1, 3) 419615
18


### In practice

- Language identification uses character n-grams; Cavnar and Trenkle (1994) described the classic n-gram frequency method that many language detectors still follow.
- The fastText classifier of Facebook AI Research represents words by their character n-grams, so that it handles morphologically rich languages and words it has never seen; its authors later published pre-trained vectors for 157 languages.
- The Google Books Ngram Viewer charts how often words and phrases (up to five-grams) appeared in millions of digitised books over time.

> **Warning.** Every additional n-gram range multiplies the number of features and the memory needed. Always combine n-grams with `min_df` (for example 2–5) or `max_features`, and check `len(vectorizer.vocabulary_)` before fitting a model.

## Sparse, high-dimensional matrices

### Concept

The document-term matrix of a real corpus is **high-dimensional**: in the case-study sample it has 213,583 columns, four times as many as there are documents, because 22 languages each bring their own vocabulary. It is also **sparse**: almost all cells are zero, because one description uses only a few dozen distinct words.

A **sparse matrix** stores only the non-zero cells. The **CSR** format (compressed sparse row), used by scikit-learn, keeps three arrays:

- `data`: the non-zero values, row by row;
- `indices`: the column number of each value;
- `indptr`: where each row starts in `data` (length = number of rows + 1).

For the three-description matrix of the bag-of-words section:

```
data    = [1 2 1 | 1 1 1 1 | 1 1 1 1 1]
indices = [6 1 5 | 5 0 7 2 | 6 0 3 4 8]
indptr  = [0, 3, 7, 12]
```

Row 0 consists of `data[0:3]` in columns `indices[0:3]` = 6, 1, 5 (toy = 1, car = 2, plastic = 1). Within a row the column numbers need not be sorted; they appear in the order in which the words were first counted. Twelve stored values replace 27 cells; for the case-study sample the saving is a factor of more than 2,000.

Word frequencies follow **Zipf's law**: a few words (`mit`, `aus`, `und`, `de`) occur extremely often, and most words occur very rarely. In the sample, half of all vocabulary words occur only once. The parameters `min_df` (drop words in fewer than k documents) and `max_df` (drop words in more than a share of documents) cut both tails and shrink the matrix.

### Why it matters

Sparsity decides which models can be used. Linear models (logistic regression, linear support vector machines, naive Bayes) work directly on sparse input and fit in seconds to minutes. Methods that need dense input or distances in all dimensions (k-nearest neighbours on raw counts, many tree ensembles, standard PCA) become slow, run out of memory, or work poorly; this is why the tree models of Session 10 are rarely applied to raw word counts. Calling `.toarray()` on the full matrix converts it to dense storage: 85 GB for the sample.

### How it works in Python

In [7]:
print(X.data, X.indices, X.indptr)   # [1 2 1 1 1 1 1 1 1 1 1 1] [6 1 5 5 0 7 2 6 0 3 4 8] [ 0  3  7 12]

cv = CountVectorizer()
D = cv.fit_transform(texts)                      # scipy.sparse CSR matrix
print(D.shape)                                   # (50000, 213583): documents x vocabulary
print(f"{D.nnz / (D.shape[0] * D.shape[1]):.5f}")  # 0.00029: 99.97 % of cells are zero
print(round(D.nnz / D.shape[0], 1))              # 61.9 distinct words per description
sparse_mb = (D.data.nbytes + D.indices.nbytes + D.indptr.nbytes) / 1e6
print(round(sparse_mb, 1), round(D.shape[0] * D.shape[1] * 8 / 1e9, 1))
# 37.4 MB as sparse matrix, 85.4 GB if stored dense with 8-byte numbers

counts = pd.Series(D.sum(axis=0).A1, index=cv.get_feature_names_out())
print(counts.nlargest(8).index.tolist())         # ['mit', 'aus', 'und', 'der', 'in', 'de', 'einem', 'die']
print(round((counts == 1).mean(), 2))            # 0.5 of all words occur only once

# min_df and max_df cut the rare and the ubiquitous words
small = CountVectorizer(min_df=5, max_df=0.5).fit(texts)
print(len(small.vocabulary_))                    # 42871 words remain

[1 2 1 1 1 1 1 1 1 1 1 1] [6 1 5 5 0 7 2 6 0 3 4 8] [ 0  3  7 12]


(50000, 213583)
0.00029
61.9
37.4 85.4
['mit', 'aus', 'und', 'der', 'in', 'de', 'einem', 'die']
0.5


42871


### In practice

- Document-term matrices are the input to classic topic models such as latent Dirichlet allocation (Blei, Ng and Jordan, 2003), used to analyse news archives and scientific literature.
- Weinberger et al. (2009) described the **hashing trick** for spam filtering at Yahoo: instead of storing a vocabulary, each word is hashed directly to one of a fixed number of columns. scikit-learn's `HashingVectorizer` implements it (see [extras notebook 02](extras/02-sklearn-hashing-vs-dict-vectorizer.ipynb)); it is used when the vocabulary is too large or keeps changing, as with new product names.
- Zipf's law (Zipf, 1949) appears in almost every language corpus and is the reason why vocabularies keep growing as more text is added; with several languages, the growth is faster still.

> **Warning.** Never call `.toarray()` or `.todense()` on a full document-term matrix, and avoid pandas `DataFrame` versions of it. Inspect a few rows (`D[:5].toarray()`) or use sparse-aware functions (`D.sum(axis=0)`, `D.nnz`).

> **Tip.** `StandardScaler` would subtract the column means and destroy sparsity. For sparse text features use `with_mean=False`, or better, use TF-IDF with row normalisation (above).

## Simple text statistics as features

### Concept

A **text statistic** is a number computed from a string without a learned vocabulary: the length in characters or tokens, the number of digits, the share of upper-case letters, or whether a fixed pattern occurs (a per-cent sign, a four-digit number). It is computed from the row alone, so it needs no `fit`.

Worked example: the description "T-shirt, knitted, 95 % cotton, 5 % elastane, for women" has 54 characters, 3 digits, 1 upper-case letter among 35 letters (share 0.03) and contains a per-cent sign.

### Why it matters

Text statistics are cheap, transparent and available for every request, and because they use only the row itself they cannot leak information from other rows. They are a quick first test of whether a text carries any signal, and they can be added as extra columns next to the TF-IDF matrix. Their limit is just as clear: a statistic *about* a text cannot tell a toy from a shoe. In the case study they describe mostly who wrote the text: the length depends on the language, and the upper-case share identifies the writing conventions of an issuing country. The per-cent sign is an exception that does carry content: textile descriptions state their fibre composition ("95 % cotton").

### How it works in Python

In [8]:
from sklearn.tree import DecisionTreeClassifier


def text_stats(text):
    """Statistics of each description; no vocabulary is learned, so nothing needs fitting."""
    t = text.fillna("")
    letters = t.map(lambda s: sum(ch.isalpha() for ch in s)).clip(lower=1)
    return pd.DataFrame({
        "n_chars": t.str.len(),
        "n_digits": t.map(lambda s: sum(ch.isdigit() for ch in s)),
        "upper_share": t.map(lambda s: sum(ch.isupper() for ch in s)) / letters,
        "has_percent": t.str.contains("%", regex=False).astype(int),
    }, index=text.index)


print(text_stats(pd.Series(["T-shirt, knitted, 95 % cotton, 5 % elastane, for women"])).round(2))
#    n_chars  n_digits  upper_share  has_percent
# 0       54         3         0.03            1

stats = text_stats(decisions["description"])
print(stats.groupby(decisions["language"]).median().loc[["de", "fr", "en"], ["n_chars", "n_digits"]])
# de 740 characters and 16 digits, fr 272 and 3, en 309 and 5: length mostly measures the language
print((stats["upper_share"] > 0.8).groupby(decisions["issuing_country"]).mean().nlargest(3).round(2).to_dict())
# {'GB': 1.0, 'IT': 0.73, 'IE': 0.69}: descriptions written almost entirely in capitals
textile = decisions["chapter"].astype(int).between(50, 63)          # section XI, textiles
print(stats["has_percent"].groupby(textile).mean().round(2).to_dict())   # {False: 0.07, True: 0.46}

# how much do the four statistics alone say about the chapter (2 digits)?
year = decisions["start_date"].dt.year
past, later = year <= 2021, year >= 2022                             # validation by time (Block 2)
tree = DecisionTreeClassifier(max_depth=10, min_samples_leaf=20, random_state=0)
tree.fit(stats[past], decisions["chapter"][past])
print(round((tree.predict(stats[later]) == decisions["chapter"][later]).mean(), 2),   # 0.19 with the statistics
      round((decisions["chapter"][later] == "85").mean(), 2))                         # 0.14 always chapter 85

   n_chars  n_digits  upper_share  has_percent
0       54         3         0.03            1


          n_chars  n_digits
language                   
de          740.0      16.0
fr          272.0       3.0
en          309.0       5.0
{'GB': 1.0, 'IT': 0.73, 'IE': 0.69}
{False: 0.07, True: 0.46}
0.19 0.14


The four statistics alone predict the right chapter for about one decision in five, against one in seven for the rule "always chapter 85" (electrical machinery, the most frequent chapter). That is a signal, but a weak one: the word counts of the previous sections carry far more information, as Block 2 shows. Statistics that describe the writer (language, country conventions) rather than the goods can also become **shortcuts**: a model that learned "capital letters → British decision → certain headings" would fail when a country changes its writing style or stops issuing decisions, as the United Kingdom did after Brexit (Session 16).

### In practice

- Readability formulas such as Flesch reading ease, built from word and sentence lengths, are used by public bodies to check that official texts are understandable; they are text statistics in the same sense.
- Spam filters used counts of capital letters, exclamation marks and links long before they used word models.

> **Tip.** To combine text statistics with TF-IDF in one model, wrap the function in a `FunctionTransformer` and join it with the vectoriser in a `ColumnTransformer` (Session 9). Scale the statistics to the range of the TF-IDF values first, for example with `MaxAbsScaler`: the length runs into the thousands, and even after `StandardScaler` a very long description gets values above 10 that outweigh all its words. On the case-study sample the four statistics scaled to [0, 1] leave the validation accuracy of the word model of Block 2 unchanged (0.772, course team's run), so for this task they are a diagnostic rather than a better feature.

> **Warning.** A statistic that differs between groups is not a cause. Descriptions with a per-cent sign are more often textiles because the tariff classifies textiles by their fibre composition, not because the sign makes a product a textile.

## Check your understanding

1. A corpus has 4 documents with 10, 12, 8 and 10 tokens and 25 types in total. How many rows and columns does its document-term matrix have, and what is the sum of all cells?
2. Why does `CountVectorizer` ignore the word "Akkuschrauber" (cordless screwdriver) in a test description if it never occurred in the training descriptions? Name one consequence for a deployed model.
3. Compute the textbook idf = ln(N / df) of a word that occurs in 500 of 50,000 descriptions, and of a word that occurs in all of them. What does scikit-learn's smoothed idf give for the second word?
4. Write down the CSR arrays (`data`, `indices`, `indptr`) for the matrix `[[0, 3], [1, 0], [0, 0]]`.
5. The upper-case share of a description predicts its issuing country well. Is it a useful feature for predicting the heading? Name one risk.

## Further reading

- Jurafsky, D. and Martin, J. H. (2026). *Speech and Language Processing*, 3rd ed. draft, chapter "Words and Tokens". https://web.stanford.edu/~jurafsky/slp3/
- Manning, C. D., Raghavan, P. and Schütze, H. (2008). *Introduction to Information Retrieval*, chapter 2 "The term vocabulary and postings lists" (including compound splitting). Cambridge University Press. https://nlp.stanford.edu/IR-book/
- Spärck Jones, K. (1972). A statistical interpretation of term specificity and its application in retrieval. *Journal of Documentation*, 28(1), 11–21. https://doi.org/10.1108/eb026526
- scikit-learn developers. *Text feature extraction* (user guide, section 6.2.3). https://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction
- World Customs Organization. *What is the Harmonized System (HS)?* https://www.wcoomd.org/en/topics/nomenclature/overview/what-is-the-harmonized-system.aspx